# 第 5 周：优化、过拟合与分类

**目标**：用一个原创非线性分类任务理解 Adam、训练/验证差距、正则化和多分类概率。  
**先修**：第 3 周逻辑回归、第 4 周验证集；需要 `numpy`、`torch`。

完成后你应该能：

- 写出 `Linear → ReLU → Linear` 的小网络；
- 正确使用 `CrossEntropyLoss`（输入 logits，标签为整数类别）；
- 区分训练表现和验证表现；
- 说明 `weight_decay` 与早停分别在做什么。


In [ ]:
import copy
import numpy as np
import torch
from torch import nn

np.random.seed(23)
torch.manual_seed(23)


## 1. 创建 XOR 风格数据

当两个坐标同号时为类别 1，否则为类别 0。单条直线无法完美分开，因此需要非线性激活。


In [ ]:
rng = np.random.default_rng(23)
x_np = rng.uniform(-1, 1, size=(600, 2)).astype("float32")
clean_labels = ((x_np[:, 0] * x_np[:, 1]) > 0).astype("int64")

order = rng.permutation(len(x_np))
train_idx, val_idx = order[:420], order[420:]
x_train = torch.tensor(x_np[train_idx])
y_train = torch.tensor(clean_labels[train_idx])
x_val = torch.tensor(x_np[val_idx])
y_val = torch.tensor(clean_labels[val_idx])

print(x_train.shape, y_train.shape, x_val.shape, y_val.shape)


## 2. 模型、指标与训练

`CrossEntropyLoss` 直接接收两个类别的 logits；不要先手动 softmax。评估时再用 softmax 查看概率。


In [ ]:
def make_model():
    return nn.Sequential(
        nn.Linear(2, 16),
        nn.ReLU(),
        nn.Linear(16, 2),
    )


def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        return (model(x).argmax(dim=1) == y).float().mean().item()


def train_model(weight_decay=0.0, epochs=300):
    torch.manual_seed(23)
    model = make_model()
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.03, weight_decay=weight_decay)
    best_state = None
    best_val = -1.0
    history = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(x_train), y_train)
        loss.backward()
        optimizer.step()

        if epoch % 25 == 0 or epoch == epochs - 1:
            train_acc = accuracy(model, x_train, y_train)
            val_acc = accuracy(model, x_val, y_val)
            history.append((epoch, loss.item(), train_acc, val_acc))
            if val_acc > best_val:
                best_val = val_acc
                best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, history


model, history = train_model(weight_decay=1e-4)
print("最后三个检查点:", history[-3:])
print("best validation accuracy:", accuracy(model, x_val, y_val))


In [ ]:
with torch.no_grad():
    logits = model(x_val[:8])
    probabilities = torch.softmax(logits, dim=1)

assert logits.shape == (8, 2)
assert probabilities.shape == (8, 2)
assert torch.allclose(probabilities.sum(dim=1), torch.ones(8), atol=1e-6)
assert accuracy(model, x_val, y_val) > 0.92
print("分类训练测试通过 ✅")


## 3. 诊断过拟合

过拟合不是“训练 loss 很低”本身，而是训练表现继续改善、验证表现却停滞或下降。  
本例数据较干净，重点是掌握诊断方式：每次实验都同时记录训练和验证指标。


In [ ]:
gaps = [(epoch, train_acc - val_acc) for epoch, _, train_acc, val_acc in history]
worst_epoch, largest_gap = max(gaps, key=lambda item: item[1])
print(f"最大训练-验证准确率差距：{largest_gap:.3f}（epoch {worst_epoch}）")


### 实操：制造并修复过拟合

任选一种修改并记录训练/验证结果：

1. 只保留 50 个训练样本；
2. 随机翻转 20% 训练标签；
3. 把隐藏层扩大到 128；
4. 比较 `weight_decay=0` 和 `1e-3`；
5. 保存验证集最好时的参数，而非最后一次参数。


## 常见错误

- 对 logits 先 softmax 再传入 `CrossEntropyLoss`；
- 标签用浮点数或 one-hot，而损失函数期待整数类别索引；
- 在验证集上反向传播；
- 根据测试集反复调参；
- 看到验证波动就不断加网络，而没有先查数据、基线和错误样本。

## 扩展与出口测验

扩展：加入第三类中心区域，修改输出维度并观察混淆。  
出口测验：解释 Adam 的作用、过拟合的证据、`weight_decay` 与早停的区别。

**通过条件**：全部断言通过，验证准确率超过 92%，完成一次受控的过拟合对照实验。
